# Trad-Chem-LLM: Fast, Repository-Compatible Google Colab Batch Generator
**Version 5** — Fully compliant with the Trad-Chem-LLM repository structure, , and .

### Overview & Non-Negotiable Output Rules
1. **One Chemical = One Folder = 4 Files**: , , , and an empty .
2. **Strict Hierarchy**: 
3. **Canonical Typed Consistency**: All three data formats represent identical database records derived from one single object model.
4. **Real Evidence**: Pulls authentic data from PubChem, ChEMBL, UniProt, RCSB PDB, and AlphaFold DB without fabricating targets or docking scores.
5. **Safe Verification**: Submits each chemical folder to the repository submission checker without .


## Step 1: One-Click Environment Setup
Installs required dependencies (, ), unpacks the complete scientific research engine, clones the official Trad-Chem-LLM repository, and configures the environment.

In [ ]:
# 1. Install dependencies
!pip install --quiet rdkit requests
import os, sys, shutil, subprocess, base64, io, tarfile

ENGINE_B64 = "..."
if not os.path.exists("tradchem_engine"):
    print("Unpacking embedded tradchem_engine package...")
    tar_data = base64.b64decode(ENGINE_B64)
    with tarfile.open(fileobj=io.BytesIO(tar_data), mode="r:gz") as tar:
        tar.extractall(".")
    print("tradchem_engine installed successfully.")

# 2. Clone official Trad-Chem-LLM repository for check_submission.py
REPO_URL = "https://github.com/INSTITUTE-OF-SCIENTIFIC-INFORMATICS/Trad-Chem-LLM.git"
REPO_DIR = "Trad-Chem-LLM"

if not os.path.exists(REPO_DIR):
    print(f"Cloning official Trad-Chem-LLM repository from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already present at {REPO_DIR}.")


## Step 2: Configure Run Parameters & Optional Context
Specify target organism, taxon ID, maximum targets, and optional traditional medicine context if omitted from the CSV.

In [ ]:
TARGET_ORGANISM = "Homo sapiens"  # @param {type:"string"}
TARGET_TAXON_ID = 9606            # @param {type:"integer"}
MAX_TARGETS = 10                  # @param {type:"slider", min:1, max:10, step:1}
INCLUDE_PREDICTIONS = True        # @param {type:"boolean"}
INCLUDE_LITERATURE = True         # @param {type:"boolean"}

DEFAULT_MEDICINE = "Harithakyadiya Kashaya"  # @param {type:"string"}
DEFAULT_SYSTEM = "Ayurveda"                  # @param ["Ayurveda", "Siddha", "Unani", "TCM", "Other"]
DEFAULT_FORMULATION = "harithakyadiya-kashaya" # @param {type:"string"}
DEFAULT_PLANT = "Curcuma longa"              # @param {type:"string"}
DEFAULT_PLANT_TYPE = "herb"                   # @param {type:"string"}
DEFAULT_PART_USED = "rhizome"                 # @param {type:"string"}
DEFAULT_FAMILY = "Zingiberaceae"              # @param {type:"string"}
DEFAULT_COMMON_NAME = "Turmeric"              # @param {type:"string"}
DEFAULT_DISEASES = "colorectal cancer; arthritis" # @param {type:"string"}

CONTEXT_CONFIG = {
    "medicine": DEFAULT_MEDICINE,
    "system": DEFAULT_SYSTEM,
    "formulation": DEFAULT_FORMULATION,
    "plant": DEFAULT_PLANT,
    "plant_type": DEFAULT_PLANT_TYPE,
    "part_used": DEFAULT_PART_USED,
    "family": DEFAULT_FAMILY,
    "common_name": DEFAULT_COMMON_NAME,
    "diseases": DEFAULT_DISEASES,
    "target_taxon_id": TARGET_TAXON_ID
}
print("Configuration loaded.")


## Step 3: Upload Input CSV & Process Batch
Upload your CSV (e.g.  with columns , , ).
The engine validates input structures, queries scientific databases, builds the canonical records, runs the submission checker, and packages the results into a downloadable ZIP archive.

In [ ]:
import io, os, csv, json, time, shutil
from google.colab import files
from tradchem_engine.csv_loader import parse_csv_file
from tradchem_engine.researcher import ChemicalResearcher
from tradchem_engine.exporters import write_chemical_folder
from tradchem_engine.checker import run_repo_checker
from tradchem_engine.reports import write_batch_summary_csv, write_source_audit_json, write_unresolved_chemicals_csv, write_input_quality_report, write_validation_results_json, write_run_configuration_json
from tradchem_engine.packaging import package_contribution_zip

uploaded = files.upload()
if not uploaded:
    print("No file uploaded. Please upload compound list updates1.csv.")
else:
    csv_filename = list(uploaded.keys())[0]
    csv_content = uploaded[csv_filename].decode("utf-8-sig", errors="replace")
    print(f"Loaded {csv_filename}. Parsing chemical rows...")
    rows, parse_warnings = parse_csv_file(csv_content, default_context=CONTEXT_CONFIG)
    print(f"Parsed {len(rows)} chemical row(s).")

    OUTPUT_DIR = "output_contributions"
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(f"{OUTPUT_DIR}/reports", exist_ok=True)

    write_input_quality_report(rows, f"{OUTPUT_DIR}/reports/input_quality_report.csv")
    write_run_configuration_json(CONTEXT_CONFIG, f"{OUTPUT_DIR}/reports/run_configuration.json")

    researcher = ChemicalResearcher()
    results = []

    print("Starting research and validation pipeline...")
    repo_root = "Trad-Chem-LLM" if os.path.exists("Trad-Chem-LLM") else None
    for idx, row in enumerate(rows, 1):
        print(f"[{idx}/{len(rows)}] Processing {row.raw_name}...")
        res = researcher.process_item(row, max_targets=MAX_TARGETS, include_predictions=INCLUDE_PREDICTIONS)
        if res.record and res.exported_targets > 0:
            folder_path = write_chemical_folder(OUTPUT_DIR, res.record)
            status, errs, warns, out_text = run_repo_checker(repo_root, folder_path)
            res.checker_status = status
            if warns:
                res.warning_or_error += " Warnings: " + "; ".join(warns)
            if errs:
                res.warning_or_error += " Errors: " + "; ".join(errs)
        results.append(res)

    write_batch_summary_csv(results, f"{OUTPUT_DIR}/reports/batch_summary.csv")
    write_source_audit_json(results, f"{OUTPUT_DIR}/reports/source_audit.json")
    write_unresolved_chemicals_csv(results, f"{OUTPUT_DIR}/reports/unresolved_chemicals.csv")
    write_validation_results_json(results, f"{OUTPUT_DIR}/reports/validation_results.json")

    zip_output = "tradchem_contributions.zip"
    package_contribution_zip(OUTPUT_DIR, zip_output)
    print(f"Batch completed! Bundled contribution archive: {zip_output}")
    print("Downloading ZIP artifact...")
    files.download(zip_output)
